# Deep Learning for Image Analysis
### YOLO Object Detection on Pascal VOC 2012

**Authors:** Bo Fu, Yehoshua Perez Condori  
**Programme:** MSc Artificial Intelligence  
**Institution:** City St George's, University of London  
**Module:** INM705 Deep Learning for Image Analysis  
**Module Leader:** Dr Riad Ibadulla  

---

### Project Overview
Implementation of YOLOv1-style object detection using VGG16 backbone trained on Pascal VOC 2012 dataset with 20 object categories.

---

### Links
- **GitHub:** https://github.com/BoFu001/YOLO-object-detection
- **Colab Notebook:** https://drive.google.com/file/d/182m9Fadqzu_SJAwi9HJrPFqUUiMgEdhU/view?usp=sharing
- **Kaggle Notebook:** https://www.kaggle.com/code/bofu001/yolo-object-detection
- **Kaggle Dataset:** https://www.kaggle.com/datasets/huanghanchina/pascal-voc-2012
- **Wandb:** https://wandb.ai/bofu001-/YOLO-VOC2012

## Setup (Colab)
Connect Google Drive so we can import the project code and save checkpoints/graphs.


In [1]:
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

Mounted at /content/drive/


## Install dependencies + set project path
We install a few libraries, then add the project folder to `sys.path` so `my_config.py` and `modules/` imports work.


In [2]:
!pip install -q torchmetrics kaggle wandb

import sys
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/Education/Deep Learning for Image Analysis/Training/YOLO-object-detection/YOLO-object-detection/')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root added to sys.path: {PROJECT_ROOT}')

# quick sanity check: project files visible
print('my_config.py exists:', (PROJECT_ROOT / 'my_config.py').exists())
print('modules/ exists:', (PROJECT_ROOT / 'modules').exists())


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 18.6 MB/s eta 0:00:00
Project root added to sys.path: /content/drive/MyDrive/Colab Notebooks/Education/Deep Learning for Image Analysis/Training/YOLO-object-detection/YOLO-object-detection
my_config.py exists: True
modules/ exists: True


In [3]:
import os
print('Current working directory:', os.getcwd())

Current working directory: /content


In [4]:

print('Drive mounted:', os.path.exists('/content/drive/MyDrive/'))
if os.path.exists('/content/drive/MyDrive/'):
    print('Contents of MyDrive:')
    print(os.listdir('/content/drive/MyDrive/')[:10])

# Let's also check if Colab Notebooks exists
if os.path.exists('/content/drive/MyDrive/Colab Notebooks/'):
    print('\nContents of Colab Notebooks:')
    print(os.listdir('/content/drive/MyDrive/Colab Notebooks/')[:10])

Drive mounted: True
Contents of MyDrive:
['Colab Notebooks', 'Crime-Type-Prediction-Using-Machine-Learning.gslides', 'Crime-Type-Prediction-Using-Machine-Learning.gvid', 'Untitled video.gvid', 'PXL_20260410_143859481.jpg', 'PXL_20260410_143857839.jpg', 'PXL_20260410_143854288.jpg', 'PXL_20260410_143852981.MP.jpg', 'PXL_20260410_084015272.jpg', 'PXL_20260410_084009367.MP.jpg']

Contents of Colab Notebooks:
['Education', '.ipynb_checkpoints']


## Optional: Weights & Biases (wandb)
If a `WANDB_API_KEY` is available in Colab Secrets we log online; otherwise we disable wandb to avoid prompts.


In [5]:
import wandb


try:
    from google.colab import userdata
    wandb_api_key = userdata.get('WANDB_API_KEY')
except Exception:
    wandb_api_key = None

if wandb_api_key:
    os.environ.pop("WANDB_MODE", None)
    wandb.login(key=wandb_api_key, relogin=False)
    print("WandB login enabled from Colab Secrets.")
else:
    os.environ["WANDB_MODE"] = "disabled"
    print("WANDB_API_KEY not found in Colab Secrets. WandB disabled, so no login prompt will appear.")
    print("Add WANDB_API_KEY to Colab Secrets if you want online WandB logging or sweeps.")


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: y-benjamin_pc (y-benjamin_pc-city-st-george-s-university-of-london) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


WandB login enabled from Colab Secrets.


## Dataset (Pascal VOC 2012)
Downloads/unzips VOC once into `/content/VOC2012/...` (skips if already present).


In [6]:
# Kaggle dataset setup for Colab local runtime
import os
from pathlib import Path

VOC_ROOT = Path('/content/VOC2012/VOC2012')
ZIP_PATH = Path('/content/pascal-voc-2012.zip')
KAGGLE_JSON = Path.home() / '.kaggle' / 'kaggle.json'

if VOC_ROOT.exists():
    print(f"Dataset already present at {VOC_ROOT}")
else:
    if not KAGGLE_JSON.exists():
        from google.colab import files
        uploaded = files.upload()
        uploaded_names = list(uploaded.keys())
        if not uploaded_names:
            raise RuntimeError("No kaggle.json uploaded.")
        first_file = uploaded_names[0]
        os.makedirs(Path.home() / '.kaggle', exist_ok=True)
        os.replace(first_file, KAGGLE_JSON)
        os.chmod(KAGGLE_JSON, 0o600)
        print(f"Kaggle credentials configured from: {first_file}")
    else:
        print("Using existing ~/.kaggle/kaggle.json")

    if not ZIP_PATH.exists():
        !kaggle datasets download -d huanghanchina/pascal-voc-2012 -p /content/
    else:
        print(f"Zip already present at {ZIP_PATH}")

    !unzip -qo /content/pascal-voc-2012.zip -d /content/VOC2012
    print("Dataset prepared at /content/VOC2012/VOC2012")

Saving kaggle.json to kaggle.json
Kaggle credentials configured from: kaggle.json
Dataset URL: https://www.kaggle.com/datasets/huanghanchina/pascal-voc-2012
License(s): DbCL-1.0
100% 3.63G/3.63G [00:22<00:00, 170MB/s]

Dataset prepared at /content/VOC2012/VOC2012


In [7]:
import random
import numpy as np
import torch
import io
import re
import json
import sys
import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt
import itertools


## Imports + reproducibility
We set seeds and deterministic flags so experiments are more repeatable.


In [8]:
from my_config import SEED, CKPT_DIR, DEVICE, IMG_DIR, ANN_DIR, CLASSES, NUM_WORKERS

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## Import project modules
These are the provided model/training/evaluation helpers we’ll call throughout the experiments.


In [9]:
# Custom Modules
from modules.Dataset import get_dataloaders
from modules.Train import train
from modules.TrainFinetune import train_finetune
from modules.TrainFinetuneLayerwise import train_finetune_layerwise
from modules.TrainFinetuneLayerwiseF1 import train_finetune_layerwise_f1
from modules.Models.YOLOv1 import YOLOv1
from modules.Models.YOLOv1Dropout import YOLOv1Dropout
from modules.Models.YOLOv1Finetune import YOLOv1Finetune
from modules.Evaluation import evaluate
from modules.Inference import inference
from contextlib import contextmanager, redirect_stdout

## Sweep helper functions
These exist only because W&B sweeps execute a callable (`train_sweep`) and our eval utility prints metrics rather than returning them.

- `reuse_existing_wandb_run()` prevents nested `wandb.init()/finish()` calls inside helper code when running under `wandb.agent`.
- `evaluate_with_capture()` captures `evaluate()` stdout so we can parse/log `mAP@0.50` and use it as the sweep metric.


In [10]:

@contextmanager
def reuse_existing_wandb_run():
    """Reuse the active sweep run inside helper functions that may call wandb.init/finish."""
    original_init = wandb.init
    original_finish = wandb.finish

    def _reuse_init(*args, **kwargs):
        return wandb.run if wandb.run is not None else original_init(*args, **kwargs)

    def _noop_finish(*args, **kwargs):
        return None

    wandb.init = _reuse_init
    wandb.finish = _noop_finish
    try:
        yield
    finally:
        wandb.init = original_init
        wandb.finish = original_finish

In [11]:
BEST_SWEEP_CKPT = None
BEST_SWEEP_RUN = None
BEST_SWEEP_MAP50 = float("-inf")
BEST_SWEEP_SUMMARY = None

In [12]:
def build_sweep_model(dropout_p):
    model = YOLOv1Finetune(S=S, B=B, C=C).to(DEVICE)

    # Apply sweep dropout knob by updating any Dropout layers in-place.
    dropout_layers = 0
    for module in model.modules():
        if isinstance(module, torch.nn.Dropout):
            module.p = float(dropout_p)
            dropout_layers += 1
    print(f"Dropout p={float(dropout_p):.3f} applied to {dropout_layers} layer(s).")

    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs!")
        model = torch.nn.DataParallel(model)

    raw_model = model.module if isinstance(model, torch.nn.DataParallel) else model
    return model, raw_model


In [13]:
def train_sweep():
    global BEST_SWEEP_CKPT, BEST_SWEEP_RUN, BEST_SWEEP_MAP50, BEST_SWEEP_SUMMARY

    run = wandb.init(project=SWEEP_PROJECT)
    config = wandb.config

    run_name = f"sweep_{run.id}"
    wandb.run.name = run_name
    print(f"Starting sweep run: {run_name}")

    # fresh loaders for each run
    train_loader, val_loader, _ = get_dataloaders(
        int(config.BATCH_SIZE),
        S, B, C,
        augment=bool(config.AUGMENT)
    )

    model, raw_model = build_sweep_model(config.DROPOUT_P)

    ckpt_path = os.path.join(CKPT_DIR, f"{run_name}.pth")

    with reuse_existing_wandb_run():
        raw_model = train_finetune_layerwise(
            model=model,
            raw_model=raw_model,
            train_loader=train_loader,
            val_loader=val_loader,
            S=S, B=B, C=C,
            BATCH_SIZE=int(config.BATCH_SIZE),
            EPOCHS=int(config.EPOCHS),
            LR_HEAD=float(config.LR_HEAD),
            LR_BACKBONE=float(config.LR_BACKBONE),
            WEIGHT_DECAY=float(config.WEIGHT_DECAY),
            LAMBDA_BOX=float(config.LAMBDA_BOX),
            LAMBDA_NOOBJ=float(config.LAMBDA_NOOBJ),
            RUN_NAME=run_name
        )

    torch.save(raw_model.state_dict(), ckpt_path)
    print(f"Sweep weights saved: {ckpt_path}")

    raw_model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    model.eval()

    metrics, eval_text = evaluate_with_capture(
        model=model,
        loader=val_loader,
        conf_thresh=float(config.CONF_THRESH),
        iou_thresh=NMS_IOU_THRESH
    )

    summary = {
        "val/mAP": metrics["val/mAP"],
        "val/mAP_50_95": metrics["val/mAP_50_95"],
        "ckpt_path": ckpt_path,
        "dropout_p": float(config.DROPOUT_P),
        "conf_thresh": float(config.CONF_THRESH),
        "augment": bool(config.AUGMENT),
    }

    wandb.log(summary)
    wandb.run.summary["ckpt_path"] = ckpt_path
    wandb.run.summary["eval_stdout"] = eval_text

    if metrics["val/mAP"] is not None and metrics["val/mAP"] > BEST_SWEEP_MAP50:
        BEST_SWEEP_MAP50 = metrics["val/mAP"]
        BEST_SWEEP_CKPT = ckpt_path
        BEST_SWEEP_RUN = run_name
        BEST_SWEEP_SUMMARY = {
            "run_name": run_name,
            "ckpt_path": ckpt_path,
            "val/mAP": metrics["val/mAP"],
            "val/mAP_50_95": metrics["val/mAP_50_95"],
            "config": dict(wandb.config),
        }

        with open(os.path.join(CKPT_DIR, "best_sweep_summary.json"), "w") as f:
            json.dump(BEST_SWEEP_SUMMARY, f, indent=2)

        print("New best sweep run found:")
        print(json.dumps(BEST_SWEEP_SUMMARY, indent=2))

    wandb.finish()

In [14]:
def evaluate_with_capture(model, loader, conf_thresh, iou_thresh):
    """Run evaluate() and capture its stdout so we can parse the printed mAP numbers."""
    buffer = io.StringIO()
    with redirect_stdout(buffer):
        _ = evaluate(
            model=model,
            test_loader=loader,
            S=S, B=B, C=C,
            conf_thresh=float(conf_thresh),
            iou_thresh=iou_thresh,
        )
    eval_text = buffer.getvalue()
    print(eval_text)

    map50_match = re.search(r"mAP@0\.50:\s*([0-9]*\.?[0-9]+)", eval_text)
    map5095_match = re.search(r"mAP@0\.50:0\.95:\s*([0-9]*\.?[0-9]+)", eval_text)

    map50 = float(map50_match.group(1)) if map50_match else None
    map5095 = float(map5095_match.group(1)) if map5095_match else None

    metrics = {"val/mAP": map50, "val/mAP_50_95": map5095}
    return metrics, eval_text


In [15]:
# YOLO parameters
S = 7
B = 2
C = 20

# training parameters
BATCH_SIZE = 16
EPOCHS = 5
LR = 1e-3
WEIGHT_DECAY = 1e-4

# loss weights
LAMBDA_BOX = 5.0
LAMBDA_NOOBJ = 0.5

# inference thresholds
CONF_THRESH = 0.30
NMS_IOU_THRESH = 0.45

In [16]:
# create all data loaders
train_loader, val_loader, test_loader = get_dataloaders(BATCH_SIZE, S, B, C)

Train set: 5717 images
Val set:   4076 images
Test set:  1747 images


#### Experiment 1 - Baseline
* Model: YOLOv1 (frozen VGG16 backbone)
* LR: 1e-3
* EPOCHS: 5
* Goal: verify model can learn and observe initial loss trend

In [17]:
# @title
RUN_NAME = "exp1_YOLOv1_lr1e-3"
CKPT_PATH = f"{CKPT_DIR}/{RUN_NAME}.pth"
EPOCHS       = 5
LR           = 1e-3

In [18]:
# @title
# create model
model = YOLOv1(S=S, B=B, C=C).to(DEVICE)

# multi-GPU support
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = torch.nn.DataParallel(model)
else :
    print("not using GPUs")

# unwrap DataParallel
raw_model = model.module if isinstance(model, torch.nn.DataParallel) else model

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 238MB/s]


not using GPUs


Training: Experiment 1

In [19]:
# @title
# training
raw_model = train(
    model=model,
    raw_model=raw_model,
    train_loader=train_loader,
    val_loader=val_loader,
    S=S, B=B, C=C,
    BATCH_SIZE=BATCH_SIZE,
    EPOCHS=EPOCHS,
    LR=LR,
    WEIGHT_DECAY=WEIGHT_DECAY,
    LAMBDA_BOX=LAMBDA_BOX,
    LAMBDA_NOOBJ=LAMBDA_NOOBJ,
    RUN_NAME=RUN_NAME
)

# save trained weights
torch.save(raw_model.state_dict(), CKPT_PATH)
print(f"trained weights saved: {RUN_NAME}.pth")

Using device: cuda


Epoch 1/5 Val: 100%|██████████| 255/255 [00:13<00:00, 18.93it/s]


Epoch 001/5 | train: 3.2245 | val: 2.6246


Epoch 2/5 Val: 100%|██████████| 255/255 [00:13<00:00, 18.93it/s]


Epoch 002/5 | train: 2.4593 | val: 2.4655


...

Epoch 004/5 | train: 2.0981 | val: 2.4869


Epoch 5/5 Val: 100%|██████████| 255/255 [00:13<00:00, 18.83it/s]

Epoch 005/5 | train: 1.9754 | val: 2.5063


box_loss,█▄▃▂▁
cls_loss,█▄▃▂▁
epoch,▁▃▅▆█
noobj_loss,█▃▂▂▁
obj_loss,█▅▃▂▁
train_loss,█▄▃▂▁
val_loss,█▁▁▂▃
box_loss,0.10046
cls_loss,1.10094
epoch,5
noobj_loss,0.46833


trained weights saved: exp1_YOLOv1_lr1e-3.pth


Evaluation: Experiment 1

In [20]:
# @title
# evaluation

# load trained weights before evaluation
raw_model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))

results = evaluate(
    model=model,
    test_loader=test_loader,
    S=S, B=B, C=C,
    conf_thresh=CONF_THRESH,
    iou_thresh=NMS_IOU_THRESH
)

mAP@0.50:      0.1048
mAP@0.50:0.95: 0.0221


#### Experiment 2 - Lower Learning Rate
* Model: YOLOv1 (frozen VGG16 backbone)
* LR: 1e-3 → 1e-4
* EPOCHS: 20
* Goal: reduce overfitting seen in Experiment 1

In [21]:
# @title
RUN_NAME   = "exp2_YOLOv1_lr1e-4"
CKPT_PATH  = f"{CKPT_DIR}/{RUN_NAME}.pth"
EPOCHS       = 20
LR           = 1e-4

In [22]:
# @title
# create model
model = YOLOv1(S=S, B=B, C=C).to(DEVICE)

# multi-GPU support
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = torch.nn.DataParallel(model)

# unwrap DataParallel
raw_model = model.module if isinstance(model, torch.nn.DataParallel) else model

Training : Experiment 2

In [ ]:
# @title
# training
raw_model = train(
    model=model,
    raw_model=raw_model,
    train_loader=train_loader,
    val_loader=val_loader,
    S=S, B=B, C=C,
    BATCH_SIZE=BATCH_SIZE,
    EPOCHS=EPOCHS,
    LR=LR,
    WEIGHT_DECAY=WEIGHT_DECAY,
    LAMBDA_BOX=LAMBDA_BOX,
    LAMBDA_NOOBJ=LAMBDA_NOOBJ,
    RUN_NAME=RUN_NAME
)

# save trained weights
torch.save(raw_model.state_dict(), CKPT_PATH)
print(f"trained weights saved: {RUN_NAME}.pth")

Evaluation: Experiment 2

In [ ]:
# @title
# evaluation

# load trained weights before evaluation
raw_model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))

results = evaluate(
    model=model,
    test_loader=test_loader,
    S=S, B=B, C=C,
    conf_thresh=CONF_THRESH,
    iou_thresh=NMS_IOU_THRESH
)

#### Experiment 3 - Dropout Regularisation
* Model: YOLOv1Dropout (frozen VGG16 backbone)
* LR: 1e-4
* EPOCHS: 20
* Dropout: p=0.5 added in head
* Goal: further reduce overfitting with dropout regularisation

In [ ]:
# @title
RUN_NAME  = "exp3_Dropout_lr1e-4"
CKPT_PATH = f"{CKPT_DIR}/{RUN_NAME}.pth"
EPOCHS = 20
LR = 1e-4

In [ ]:
# @title
# create model
model = YOLOv1Dropout(S=S, B=B, C=C).to(DEVICE)

# multi-GPU support
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = torch.nn.DataParallel(model)

# unwrap DataParallel
raw_model = model.module if isinstance(model, torch.nn.DataParallel) else model

Training: Experiment 3

In [ ]:
# @title

# training
raw_model = train(
    model=model,
    raw_model=raw_model,
    train_loader=train_loader,
    val_loader=val_loader,
    S=S, B=B, C=C,
    BATCH_SIZE=BATCH_SIZE,
    EPOCHS=EPOCHS, LR=LR,
    WEIGHT_DECAY=WEIGHT_DECAY,
    LAMBDA_BOX=LAMBDA_BOX,
    LAMBDA_NOOBJ=LAMBDA_NOOBJ,
    RUN_NAME=RUN_NAME
)
# save trained weights
torch.save(raw_model.state_dict(), CKPT_PATH)
print(f"trained weights saved: {RUN_NAME}.pth")


Evaluation: Experiment 3

In [ ]:
# @title
# evaluation
raw_model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
results = evaluate(
    model=model,
    test_loader=test_loader,
    S=S, B=B, C=C,
    conf_thresh=CONF_THRESH,
    iou_thresh=NMS_IOU_THRESH
)

#### Experiment 4 - VGG16 Fine-tuning with Early Stopping
* Model: YOLOv1Finetune (last 2 conv layers of VGG16 unfrozen)
* LR: 1e-4
* EPOCHS: 20 (Early Stopping: patience=5)
* Dropout: p=0.5 in head
* Goal: improve feature extraction by fine-tuning backbone on VOC dataset

In [ ]:
# @title
RUN_NAME  = "exp4_Finetune_lr1e-4"
CKPT_PATH = f"{CKPT_DIR}/{RUN_NAME}.pth"
EPOCHS = 20
LR = 1e-4

In [ ]:
# @title
# create model
model = YOLOv1Finetune(S=S, B=B, C=C).to(DEVICE)

# multi-GPU support
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = torch.nn.DataParallel(model)

# unwrap DataParallel
raw_model = model.module if isinstance(model, torch.nn.DataParallel) else model

Training: Experiment 4

In [ ]:
# @title
# training
raw_model = train_finetune(
    model=model,
    raw_model=raw_model,
    train_loader=train_loader,
    val_loader=val_loader,
    S=S, B=B, C=C,
    BATCH_SIZE=BATCH_SIZE,
    EPOCHS=EPOCHS,
    LR=LR,
    WEIGHT_DECAY=WEIGHT_DECAY,
    LAMBDA_BOX=LAMBDA_BOX,
    LAMBDA_NOOBJ=LAMBDA_NOOBJ,
    RUN_NAME=RUN_NAME
)
# save trained weights
torch.save(raw_model.state_dict(), CKPT_PATH)
print(f"trained weights saved: {RUN_NAME}.pth")

Evaluation: Experiment 4

In [ ]:
# @title
# evaluation
raw_model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
results = evaluate(
    model=model,
    test_loader=test_loader,
    S=S, B=B, C=C,
    conf_thresh=CONF_THRESH,
    iou_thresh=NMS_IOU_THRESH
)

#### Experiment 5 - Layer-wise Learning Rate
* Model: YOLOv1Finetune (last 2 conv layers of VGG16 unfrozen)
* LR head: 1e-4
* LR backbone: 1e-5
* EPOCHS: 20 (Early Stopping: patience=5)
* Dropout: p=0.5 in head
* Goal: more stable fine-tuning with smaller backbone LR

In [ ]:
# @title
RUN_NAME  = "exp5_Finetune_lrH1e-4_lrB1e-5"
CKPT_PATH = f"{CKPT_DIR}/{RUN_NAME}.pth"
EPOCHS = 20
LR_HEAD=1e-4
LR_BACKBONE=1e-5

In [ ]:
# create model
model = YOLOv1Finetune(S=S, B=B, C=C).to(DEVICE)

# multi-GPU support
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = torch.nn.DataParallel(model)

# unwrap DataParallel
raw_model = model.module if isinstance(model, torch.nn.DataParallel) else model


Training: Experiment 5

In [ ]:
# training
raw_model = train_finetune_layerwise(
    model=model,
    raw_model=raw_model,
    train_loader=train_loader,
    val_loader=val_loader,
    S=S, B=B, C=C,
    BATCH_SIZE=BATCH_SIZE,
    EPOCHS=EPOCHS,
    LR_HEAD=LR_HEAD,
    LR_BACKBONE=LR_BACKBONE,
    WEIGHT_DECAY=WEIGHT_DECAY,
    LAMBDA_BOX=LAMBDA_BOX,
    LAMBDA_NOOBJ=LAMBDA_NOOBJ,
    RUN_NAME=RUN_NAME
)
# save trained weights
torch.save(raw_model.state_dict(), CKPT_PATH)
print(f"trained weights saved: {RUN_NAME}.pth")

Evaluation: Experiment 5

In [ ]:
# @title
# evaluation
raw_model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
results = evaluate(
    model=model,
    test_loader=test_loader,
    S=S, B=B, C=C,
    conf_thresh=CONF_THRESH,
    iou_thresh=NMS_IOU_THRESH
)

#### Experiment 6 - Layer-wise LR Tuning
* Model: YOLOv1Finetune (last 2 conv layers of VGG16 unfrozen)
* LR head: 1e-4
* LR backbone: 5e-5 (increased from exp5)
* EPOCHS: 20 (Early Stopping: patience=5)
* Dropout: p=0.5 in head
* Goal: find better backbone LR between exp4 (1e-4) and exp5 (1e-5)

In [ ]:
RUN_NAME = "exp6_Finetune_lrH1e-4_lrB5e-5"
CKPT_PATH = f"{CKPT_DIR}/{RUN_NAME}.pth"
EPOCHS = 20
LR_HEAD=1e-4
LR_BACKBONE= 5e-5

In [ ]:
# @title
# create model
model = YOLOv1Finetune(S=S, B=B, C=C).to(DEVICE)

# multi-GPU support
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = torch.nn.DataParallel(model)

# unwrap DataParallel
raw_model = model.module if isinstance(model, torch.nn.DataParallel) else model


Training: Experiment 6

In [ ]:
# @title
# training
raw_model = train_finetune_layerwise(
    model = model,
    raw_model = raw_model,
    train_loader = train_loader,
    val_loader = val_loader,
    S=S, B=B, C=C,
    BATCH_SIZE = BATCH_SIZE,
    EPOCHS = EPOCHS,
    LR_HEAD = LR_HEAD,
    LR_BACKBONE = LR_BACKBONE,
    WEIGHT_DECAY = WEIGHT_DECAY,
    LAMBDA_BOX = LAMBDA_BOX,
    LAMBDA_NOOBJ = LAMBDA_NOOBJ,
    RUN_NAME = RUN_NAME
)
# save trained weights
torch.save(raw_model.state_dict(), CKPT_PATH)
print(f"trained weights saved: {RUN_NAME}.pth")


Evluation: Experiment 6

In [ ]:
# evaluation
raw_model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
results = evaluate(
    model=model,
    test_loader=test_loader,
    S=S, B=B, C=C,
    conf_thresh=CONF_THRESH,
    iou_thresh=NMS_IOU_THRESH
)

#### Experiment 7 - Data Augmentation
* Model: YOLOv1Finetune (last 2 conv layers of VGG16 unfrozen)
* LR: 1e-4
* EPOCHS: 20 (Early Stopping: patience=5)
* Dropout: p=0.5 in head
* Augmentation: ColorJitter (brightness, contrast, saturation, hue)
* Goal: reduce overfitting with colour augmentation on training set

In [ ]:
RUN_NAME = "exp7_Finetune_Aug_lr1e-4"
CKPT_PATH = f"{CKPT_DIR}/{RUN_NAME}.pth"
EPOCHS = 20
LR = 1e-4

In [ ]:
# create data loaders with augmentation
train_loader, val_loader, test_loader = get_dataloaders(BATCH_SIZE, S, B, C, augment=True)

In [ ]:
# @title
# create model
model = YOLOv1Finetune(S=S, B=B, C=C).to(DEVICE)

# multi-GPU support
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = torch.nn.DataParallel(model)

# unwrap DataParallel
raw_model = model.module if isinstance(model, torch.nn.DataParallel) else model


Training: Experiment 7

In [ ]:
# @title
# training
raw_model = train_finetune(
    model=model,
    raw_model=raw_model,
    train_loader=train_loader,
    val_loader=val_loader,
    S=S, B=B, C=C,
    BATCH_SIZE=BATCH_SIZE,
    EPOCHS=EPOCHS,
    LR=LR,
    WEIGHT_DECAY=WEIGHT_DECAY,
    LAMBDA_BOX=LAMBDA_BOX,
    LAMBDA_NOOBJ=LAMBDA_NOOBJ,
    RUN_NAME=RUN_NAME
)
# save trained weights
torch.save(raw_model.state_dict(), CKPT_PATH)
print(f"trained weights saved: {RUN_NAME}.pth")


Evaluation: Experiment 7

In [ ]:
# @title
# evaluation
raw_model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
results = evaluate(
    model=model,
    test_loader=test_loader,
    S=S, B=B, C=C,
    conf_thresh=CONF_THRESH,
    iou_thresh=NMS_IOU_THRESH
)

#### Experiment 8 - Automated Hyperparameter Tuning with WandB Sweeps
This section adds a **Bayesian WandB Sweep** on top of the strongest manual setup:
* Model: `YOLOv1Finetune`
* Optimiser style: **layer-wise learning rates**
* Sweep metric: **validation mAP@0.50**
* Search space: `LR_HEAD`, `LR_BACKBONE`, `WEIGHT_DECAY`, `LAMBDA_BOX`, `LAMBDA_NOOBJ`, `DROPOUT_P`, `CONF_THRESH`

Notes:
* This reuses the existing `train_finetune_layerwise()` training function.
* A small WandB patch is included so the sweep run is reused even if your training helpers already call `wandb.init()` or `wandb.finish()`.
* `DROPOUT_P` is applied by updating any `torch.nn.Dropout` layers found in the model.

Weights and Biases Sweep: Results

In [ ]:
# @title

SWEEP_PROJECT = "yolo-object-detection"
# How many runs (trials) the agent will execute. Increase for better search, decrease for time/compute limits.
SWEEP_COUNT = 20  # reduce this if Colab time is tight

sweep_config = {
    # Bayesian optimisation over the parameters below
    "method": "bayes",

    # The scalar to maximise across sweep trials
    "metric": {"name": "val/mAP", "goal": "maximize"},

    "parameters": {
        # Fixed knobs (kept constant across all trials)
        "EPOCHS": {"value": 20},
        "BATCH_SIZE": {"value": BATCH_SIZE},
        "AUGMENT": {"value": False},

        # Learning rates: search on a log scale (LRs typically vary by orders of magnitude)
        "LR_HEAD": {
            "min": 1e-5,
            "max": 1e-3,
            "distribution": "log_uniform_values",
        },
        "LR_BACKBONE": {
            "min": 1e-6,
            "max": 1e-4,
            "distribution": "log_uniform_values",
        },

        # Weight decay: also varies best on a log scale
        "WEIGHT_DECAY": {
            "min": 1e-5,
            "max": 1e-3,
            "distribution": "log_uniform_values",
        },

        # YOLO loss weights: linear ranges are OK (these are already in human-scale ranges)
        "LAMBDA_BOX": {
            "min": 2.0,
            "max": 10.0,
        },
        "LAMBDA_NOOBJ": {
            "min": 0.1,
            "max": 1.0,
        },

        # Dropout probability applied to any `torch.nn.Dropout` layers found in the model
        # (If the model has no Dropout layers, this won't change anything.)
        "DROPOUT_P": {
            "min": 0.3,
            "max": 0.7,
        },

        # Confidence threshold used during evaluation to filter predictions.
        # NOTE: This is an *evaluation-time* knob, not training-time. Optimising it can inflate mAP by tuning the
        # decision threshold; keep it fixed if you want strict apples-to-apples model comparisons.
        "CONF_THRESH": {
            "min": 0.2,
            "max": 0.5,
        },
    },
}

print(json.dumps(sweep_config, indent=2))

In [ ]:
# @title
sweep_id = wandb.sweep(sweep_config, project=SWEEP_PROJECT)
print(f"Sweep ID: {sweep_id}")
wandb.agent(sweep_id, function=train_sweep, count=SWEEP_COUNT)

print("\nBest sweep summary:")
print(json.dumps(BEST_SWEEP_SUMMARY, indent=2) if BEST_SWEEP_SUMMARY else "No successful sweep result captured.")

### Inspect sweep results (by sweep id)
If you know the sweep path (`entity/project/sweep_id`), export configs + metrics into a single CSV.


In [ ]:
# @title
import pandas as pd
import wandb

api = wandb.Api()

# replace with your actual values
sweep = api.sweep("y-benjamin_pc-city-st-george-s-university-of-london/yolo-object-detection/8ljoiuem")

rows = []
for run in sweep.runs:
    cfg = {k: v for k, v in run.config.items() if not k.startswith("_")}
    summ = dict(run.summary)

    rows.append({
        "run_name": run.name,
        "state": run.state,
        "val/mAP": summ.get("val/mAP"),
        "val/mAP_50_95": summ.get("val/mAP_50_95"),
        "train/loss": summ.get("train/loss"),
        "val/loss": summ.get("val/loss"),
        "precision": summ.get("val/precision"),
        "recall": summ.get("val/recall"),
        "LR_HEAD": cfg.get("LR_HEAD"),
        "LR_BACKBONE": cfg.get("LR_BACKBONE"),
        "LAMBDA_BOX": cfg.get("LAMBDA_BOX"),
        "LAMBDA_NOOBJ": cfg.get("LAMBDA_NOOBJ"),
        "WEIGHT_DECAY": cfg.get("WEIGHT_DECAY"),
        "DROPOUT_P": cfg.get("DROPOUT_P"),
        "CONF_THRESH": cfg.get("CONF_THRESH"),
        "BATCH_SIZE": cfg.get("BATCH_SIZE"),
        "AUGMENT": cfg.get("AUGMENT"),
    })

df = pd.DataFrame(rows)

# best by val/mAP
df_map = df.sort_values("val/mAP", ascending=False)

# best by stricter localisation metric
df_map5095 = df.sort_values("val/mAP_50_95", ascending=False)

print("Top 10 by val/mAP")
print(df_map.head(10).to_string(index=False))

print("\nTop 10 by val/mAP_50_95")
print(df_map5095.head(10).to_string(index=False))

df.to_csv("sweep_results_full.csv", index=False)

In [ ]:
# @title
import wandb
import pandas as pd

ENTITY = "y-benjamin_pc-city-st-george-s-university-of-london"
PROJECT = "yolo-object-detection"
TOP_K = 10

api = wandb.Api()
runs = api.runs(f"{ENTITY}/{PROJECT}")

rows = []

for run in runs:
    summary = run.summary or {}
    config = run.config or {}

    rows.append({
        "run_name": run.name,
        "state": run.state,
        "val/mAP": summary.get("val/mAP"),
        "val/mAP_50_95": summary.get("val/mAP_50_95"),
        "train/loss": summary.get("train/loss"),
        "val/loss": summary.get("val/loss"),
        "precision": summary.get("precision"),
        "recall": summary.get("recall"),
        "LR_HEAD": config.get("LR_HEAD"),
        "LR_BACKBONE": config.get("LR_BACKBONE"),
        "LAMBDA_BOX": config.get("LAMBDA_BOX"),
        "LAMBDA_NOOBJ": config.get("LAMBDA_NOOBJ"),
        "WEIGHT_DECAY": config.get("WEIGHT_DECAY"),
        "DROPOUT_P": config.get("DROPOUT_P"),
        "CONF_THRESH": config.get("CONF_THRESH"),
        "BATCH_SIZE": config.get("BATCH_SIZE"),
        "AUGMENT": config.get("AUGMENT"),
    })

df = pd.DataFrame(rows)

# top runs by val/mAP
top_map = (
    df.dropna(subset=["val/mAP"])
      .sort_values("val/mAP", ascending=False)
      .head(TOP_K)
)

print(f"Top {TOP_K} by val/mAP")
print(top_map.to_string(index=False))

# top runs by val/mAP_50_95
top_map5095 = (
    df.dropna(subset=["val/mAP_50_95"])
      .sort_values("val/mAP_50_95", ascending=False)
      .head(TOP_K)
)

print("\n" + "="*100 + "\n")
print(f"Top {TOP_K} by val/mAP_50_95")
print(top_map5095.to_string(index=False))

### Export run histories + plots
This downloads each run’s history and saves `history.csv` plus a few simple PNG plots to Drive.


In [ ]:
# @title
import wandb
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

ENTITY = "y-benjamin_pc-city-st-george-s-university-of-london"
PROJECT = "yolo-object-detection"

OUTPUT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/Education/Deep Learning for Image Analysis/YOLO-object-detection/graphs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LOSS_METRICS = [
    "train_loss",
    "val_loss",
    "box_loss",
    "obj_loss",
    "noobj_loss",
    "cls_loss",
]

RUN_NAME_FILTER = None

SAVE_CSV = True

def safe_filename(text: str) -> str:
    bad = '<>:"/\\|?*'
    for ch in bad:
        text = text.replace(ch, '_')
    return text.strip().replace(' ', '_')

def get_epoch_or_step(df: pd.DataFrame) -> pd.Series:
    if 'epoch' in df.columns:
        return df['epoch']
    if '_step' in df.columns:
        return df['_step']
    return pd.Series(df.index, index=df.index)

api = wandb.Api()
runs = api.runs(f"{ENTITY}/{PROJECT}")

summary_rows = []
processed = 0

for run in runs:
    if RUN_NAME_FILTER and RUN_NAME_FILTER.lower() not in (run.name or '').lower():
        continue

    print(f"[INFO] Processing run: {run.name} ({run.id})")

    try:
        df = run.history(samples=100000)
    except Exception as e:
        print(f"[SKIP] Could not load history for {run.name}: {e}")
        continue

    if df is None or df.empty:
        print(f"[SKIP] No history for {run.name}")
        continue

    run_slug = safe_filename(f"{run.name}_{run.id}")
    run_dir = OUTPUT_DIR / run_slug
    run_dir.mkdir(parents=True, exist_ok=True)

    if SAVE_CSV:
        df.to_csv(run_dir / 'history.csv', index=False)

    x = get_epoch_or_step(df)

    for metric in LOSS_METRICS:
        if metric not in df.columns:
            continue

        metric_df = pd.DataFrame({'x': x, 'y': df[metric]}).dropna()
        if metric_df.empty:
            continue

        plt.figure(figsize=(8, 5))
        plt.plot(metric_df['x'], metric_df['y'])
        plt.xlabel('Epoch' if 'epoch' in df.columns else 'Step')
        plt.ylabel(metric)
        plt.title(f"{run.name} - {metric}")
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(run_dir / f"{metric}.png", dpi=200)
        plt.close()

    if 'train_loss' in df.columns and 'val_loss' in df.columns:
        pair_df = pd.DataFrame({'x': x, 'train_loss': df['train_loss'], 'val_loss': df['val_loss']}).dropna(how='all')
        if not pair_df.empty:
            plt.figure(figsize=(8, 5))
            if pair_df['train_loss'].notna().any():
                plt.plot(pair_df['x'], pair_df['train_loss'], label='train_loss')
            if pair_df['val_loss'].notna().any():
                plt.plot(pair_df['x'], pair_df['val_loss'], label='val_loss')
            plt.xlabel('Epoch' if 'epoch' in df.columns else 'Step')
            plt.ylabel('Loss')
            plt.title(f"{run.name} - Train vs Validation Loss")
            plt.legend()
            plt.grid(True)
            plt.tight_layout()
            plt.savefig(run_dir / 'train_vs_val_loss.png', dpi=200)
            plt.close()

    summary = run.summary or {}
    row = {
        'run_name': run.name,
        'run_id': run.id,
        'state': getattr(run, 'state', None),
        'url': getattr(run, 'url', None),
    }

    candidate_keys = [
        'val_loss',
        'train_loss',
        'box_loss',
        'obj_loss',
        'noobj_loss',
        'cls_loss',
        'val/mAP',
        'val/mAP50',
        'metrics/mAP50',
        'mAP@0.50',
        'mAP50',
    ]

    for key in candidate_keys:
        row[key] = summary.get(key, None)

    for metric in LOSS_METRICS:
        if row.get(metric) is None and metric in df.columns:
            non_null = df[metric].dropna()
            row[metric] = non_null.iloc[-1] if not non_null.empty else None

    summary_rows.append(row)
    processed += 1

if not summary_rows:
    print('[DONE] No runs matched (or no histories were available).')
else:
    summary_df = pd.DataFrame(summary_rows)
    summary_path = OUTPUT_DIR / 'run_summary.csv'
    summary_df.to_csv(summary_path, index=False)

    for metric in ['val_loss', 'train_loss', 'val/mAP', 'val/mAP50', 'mAP@0.50', 'mAP50']:
        if metric not in summary_df.columns:
            continue

        temp = summary_df[['run_name', metric]].dropna()
        if temp.empty:
            continue

        temp = temp.sort_values(by=metric, ascending=True)
        plt.figure(figsize=(10, max(5, len(temp) * 0.35)))
        plt.barh(temp['run_name'], temp[metric])
        plt.xlabel(metric)
        plt.ylabel('Run')
        plt.title(f"{metric} across runs")
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / f"compare_{safe_filename(metric)}.png", dpi=200)
        plt.close()

    print(f"[DONE] Processed {processed} runs.")
    print(f"[DONE] Output saved to: {OUTPUT_DIR.resolve()}")


#### Experiment 9 — Final model selection + seeded retraining
We take the best sweep configuration, evaluate its checkpoint directly, then retrain from scratch across a few seeds to check stability.


In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch
import importlib

import modules.TrainFinetuneLayerwiseF1 as tflf1
importlib.reload(tflf1)

from modules.TrainFinetuneLayerwiseF1 import (
    train_finetune_layerwise_f1,
    evaluate_detection_f1,
)

FINAL_RETRAIN_SEEDS = [SEED, SEED + 1, SEED + 2]

TARGET_SWEEP_RUN = "sweep_8nxmzivh"
SELECTED_SWEEP_CKPT = os.path.join(CKPT_DIR, f"{TARGET_SWEEP_RUN}.pth")

FINAL_SWEEP_CONFIG = {
    "LR_HEAD": 8.106336470374642e-05,
    "LR_BACKBONE": 1.569618297387051e-05,
    "WEIGHT_DECAY": 4.538826458245303e-04,
    "LAMBDA_BOX": 6.068012807763245,
    "LAMBDA_NOOBJ": 0.3662524566728407,
    "DROPOUT_P": 0.3435466398976728,
    "CONF_THRESH": 0.20561785961650125,
    "BATCH_SIZE": 16,
    "AUGMENT": False,
    "EPOCHS": 20,
    "F1_IOU_MATCH_THRESH": 0.50,
}

FINAL_RESULTS_CSV = os.path.join(CKPT_DIR, "exp9_final_retrain_results_f1.csv")
FINAL_SUMMARY_JSON = os.path.join(CKPT_DIR, "exp9_final_retrain_summary_f1.json")

BEST_FINAL_CKPT = None
BEST_FINAL_RUN_NAME = None
BEST_FINAL_SUMMARY = None
FINAL_RETRAIN_RESULTS = []

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def get_eval_bundle(model, loader, conf_thresh, nms_iou_thresh, f1_iou_match_thresh):
    map_metrics, _ = evaluate_with_capture(
        model=model,
        loader=loader,
        conf_thresh=conf_thresh,
        iou_thresh=nms_iou_thresh
    )

    f1_metrics = evaluate_detection_f1(
        model=model,
        loader=loader,
        S=S, B=B, C=C,
        conf_thresh=conf_thresh,
        iou_thresh=nms_iou_thresh,
        match_iou_thresh=f1_iou_match_thresh
    )

    return {
        "mAP": map_metrics.get("val/mAP"),
        "mAP_50_95": map_metrics.get("val/mAP_50_95"),
        "tp": f1_metrics.get("tp"),
        "fp": f1_metrics.get("fp"),
        "fn": f1_metrics.get("fn"),
        "precision": f1_metrics.get("precision"),
        "recall": f1_metrics.get("recall"),
        "f1": f1_metrics.get("f1"),
    }

print("Selected sweep checkpoint:", SELECTED_SWEEP_CKPT)
print("Using exact sweep config:")
print(FINAL_SWEEP_CONFIG)

if not os.path.exists(SELECTED_SWEEP_CKPT):
    raise FileNotFoundError(f"Checkpoint not found: {SELECTED_SWEEP_CKPT}")

set_all_seeds(SEED)

train_loader, val_loader, test_loader = get_dataloaders(
    FINAL_SWEEP_CONFIG["BATCH_SIZE"],
    S, B, C,
    augment=FINAL_SWEEP_CONFIG["AUGMENT"]
)

print("\nEvaluating selected sweep checkpoint directly (no retraining)")

model, raw_model = build_sweep_model(FINAL_SWEEP_CONFIG["DROPOUT_P"])
raw_model.load_state_dict(torch.load(SELECTED_SWEEP_CKPT, map_location=DEVICE))
model.eval()

selected_val_eval = get_eval_bundle(
    model=model,
    loader=val_loader,
    conf_thresh=FINAL_SWEEP_CONFIG["CONF_THRESH"],
    nms_iou_thresh=NMS_IOU_THRESH,
    f1_iou_match_thresh=FINAL_SWEEP_CONFIG["F1_IOU_MATCH_THRESH"],
)

selected_test_eval = get_eval_bundle(
    model=model,
    loader=test_loader,
    conf_thresh=FINAL_SWEEP_CONFIG["CONF_THRESH"],
    nms_iou_thresh=NMS_IOU_THRESH,
    f1_iou_match_thresh=FINAL_SWEEP_CONFIG["F1_IOU_MATCH_THRESH"],
)

print("Selected checkpoint metrics:")
print({
    "val/mAP": selected_val_eval["mAP"],
    "val/mAP_50_95": selected_val_eval["mAP_50_95"],
    "val/F1": selected_val_eval["f1"],
    "val/precision": selected_val_eval["precision"],
    "val/recall": selected_val_eval["recall"],
    "test/mAP": selected_test_eval["mAP"],
    "test/mAP_50_95": selected_test_eval["mAP_50_95"],
    "test/F1": selected_test_eval["f1"],
    "test/precision": selected_test_eval["precision"],
    "test/recall": selected_test_eval["recall"],
})

print("\nRetraining from scratch across seeds using exact selected hyperparameters")

for seed in FINAL_RETRAIN_SEEDS:
    print(f"\nRetraining with seed {seed}")

    set_all_seeds(seed)

    train_loader, val_loader, test_loader = get_dataloaders(
        FINAL_SWEEP_CONFIG["BATCH_SIZE"],
        S, B, C,
        augment=FINAL_SWEEP_CONFIG["AUGMENT"]
    )

    model, raw_model = build_sweep_model(FINAL_SWEEP_CONFIG["DROPOUT_P"])

    run_name = f"exp9_seed{seed}"
    ckpt_path = os.path.join(CKPT_DIR, f"{run_name}.pth")

    raw_model = train_finetune_layerwise_f1(
        model=model,
        raw_model=raw_model,
        train_loader=train_loader,
        val_loader=val_loader,
        S=S, B=B, C=C,
        BATCH_SIZE=FINAL_SWEEP_CONFIG["BATCH_SIZE"],
        EPOCHS=FINAL_SWEEP_CONFIG["EPOCHS"],
        LR_HEAD=FINAL_SWEEP_CONFIG["LR_HEAD"],
        LR_BACKBONE=FINAL_SWEEP_CONFIG["LR_BACKBONE"],
        WEIGHT_DECAY=FINAL_SWEEP_CONFIG["WEIGHT_DECAY"],
        LAMBDA_BOX=FINAL_SWEEP_CONFIG["LAMBDA_BOX"],
        LAMBDA_NOOBJ=FINAL_SWEEP_CONFIG["LAMBDA_NOOBJ"],
        RUN_NAME=run_name,
        CONF_THRESH=FINAL_SWEEP_CONFIG["CONF_THRESH"],
        NMS_IOU_THRESH=NMS_IOU_THRESH,
        F1_IOU_MATCH_THRESH=FINAL_SWEEP_CONFIG["F1_IOU_MATCH_THRESH"],
    )

    torch.save(raw_model.state_dict(), ckpt_path)

    raw_model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    model.eval()

    val_eval = get_eval_bundle(
        model=model,
        loader=val_loader,
        conf_thresh=FINAL_SWEEP_CONFIG["CONF_THRESH"],
        nms_iou_thresh=NMS_IOU_THRESH,
        f1_iou_match_thresh=FINAL_SWEEP_CONFIG["F1_IOU_MATCH_THRESH"],
    )

    test_eval = get_eval_bundle(
        model=model,
        loader=test_loader,
        conf_thresh=FINAL_SWEEP_CONFIG["CONF_THRESH"],
        nms_iou_thresh=NMS_IOU_THRESH,
        f1_iou_match_thresh=FINAL_SWEEP_CONFIG["F1_IOU_MATCH_THRESH"],
    )

    row = {
        "run_name": run_name,
        "seed": seed,
        "ckpt_path": ckpt_path,
        "val/mAP": val_eval["mAP"],
        "val/mAP_50_95": val_eval["mAP_50_95"],
        "val/precision": val_eval["precision"],
        "val/recall": val_eval["recall"],
        "val/F1": val_eval["f1"],
        "val/TP": val_eval["tp"],
        "val/FP": val_eval["fp"],
        "val/FN": val_eval["fn"],
        "test/mAP": test_eval["mAP"],
        "test/mAP_50_95": test_eval["mAP_50_95"],
        "test/precision": test_eval["precision"],
        "test/recall": test_eval["recall"],
        "test/F1": test_eval["f1"],
        "test/TP": test_eval["tp"],
        "test/FP": test_eval["fp"],
        "test/FN": test_eval["fn"],
    }

    FINAL_RETRAIN_RESULTS.append(row)
    print("Finished:", row)

results_df = pd.DataFrame(FINAL_RETRAIN_RESULTS)
display(results_df)

if results_df.empty:
    raise RuntimeError("No final retraining results were collected.")

results_df.to_csv(FINAL_RESULTS_CSV, index=False)

results_df_sorted = results_df.sort_values(
    by=["val/F1", "val/mAP"],
    ascending=[False, False]
).reset_index(drop=True)

BEST_FINAL_SUMMARY = results_df_sorted.iloc[0].to_dict()
BEST_FINAL_CKPT = BEST_FINAL_SUMMARY["ckpt_path"]
BEST_FINAL_RUN_NAME = BEST_FINAL_SUMMARY["run_name"]

summary_payload = {
    "selected_sweep_run": TARGET_SWEEP_RUN,
    "selected_sweep_ckpt": SELECTED_SWEEP_CKPT,
    "selected_checkpoint_direct_eval": {
        "val/mAP": selected_val_eval["mAP"],
        "val/mAP_50_95": selected_val_eval["mAP_50_95"],
        "val/F1": selected_val_eval["f1"],
        "val/precision": selected_val_eval["precision"],
        "val/recall": selected_val_eval["recall"],
        "test/mAP": selected_test_eval["mAP"],
        "test/mAP_50_95": selected_test_eval["mAP_50_95"],
        "test/F1": selected_test_eval["f1"],
        "test/precision": selected_test_eval["precision"],
        "test/recall": selected_test_eval["recall"],
    },
    "final_sweep_config": FINAL_SWEEP_CONFIG,
    "selection_rule": "best by val/F1, tie-break by val/mAP",
    "best_final_run_name": BEST_FINAL_RUN_NAME,
    "best_final_ckpt": BEST_FINAL_CKPT,
    "best_final_summary": BEST_FINAL_SUMMARY,
    "mean_val_mAP": float(results_df["val/mAP"].mean()),
    "std_val_mAP": float(results_df["val/mAP"].std(ddof=1)) if len(results_df) > 1 else 0.0,
    "mean_val_F1": float(results_df["val/F1"].mean()),
    "std_val_F1": float(results_df["val/F1"].std(ddof=1)) if len(results_df) > 1 else 0.0,
    "mean_test_mAP": float(results_df["test/mAP"].mean()),
    "std_test_mAP": float(results_df["test/mAP"].std(ddof=1)) if len(results_df) > 1 else 0.0,
    "mean_test_F1": float(results_df["test/F1"].mean()),
    "std_test_F1": float(results_df["test/F1"].std(ddof=1)) if len(results_df) > 1 else 0.0,
}

with open(FINAL_SUMMARY_JSON, "w") as f:
    json.dump(summary_payload, f, indent=2)

print("\nDirect selected checkpoint evaluation:")
print(summary_payload["selected_checkpoint_direct_eval"])

print("\nFinal seeded retraining results:")
display(results_df_sorted)

print("\nBest final retrained run:")
print(BEST_FINAL_SUMMARY)

print("\nSaved:")
print("CSV  ->", FINAL_RESULTS_CSV)
print("JSON ->", FINAL_SUMMARY_JSON)
print("Best checkpoint ->", BEST_FINAL_CKPT)

#### Load final checkpoint for inference
Pick the checkpoint you want to carry into threshold tuning and qualitative inference.


In [ ]:
# @title
FINAL_CKPT = f"/content/drive/MyDrive/Colab Notebooks/Education/Deep Learning for Image Analysis/YOLO-object-detection/checkpoints/exp9_seed42.pth"

if FINAL_CKPT is None or not os.path.exists(FINAL_CKPT):
    raise FileNotFoundError(f"Checkpoint not found: {FINAL_CKPT}")

print("Using checkpoint:", FINAL_CKPT)


In [ ]:
# @title
FINAL_INFER_CONFIG = {
    "DROPOUT_P": 0.3435466398976728,
    "CONF_THRESH": 0.20561785961650125,
    "NMS_IOU_THRESH": 0.45,
    "BATCH_SIZE": 16,
    "AUGMENT": False,
}


In [ ]:
model, raw_model = build_sweep_model(FINAL_INFER_CONFIG["DROPOUT_P"])
raw_model.load_state_dict(torch.load(FINAL_CKPT, map_location=DEVICE))
model.eval()


In [ ]:
# Sanity checks

print("Device:", DEVICE)
print("Model eval mode:", not model.training)
print("CONF_THRESH:", FINAL_INFER_CONFIG["CONF_THRESH"])
print("NMS_IOU_THRESH:", FINAL_INFER_CONFIG["NMS_IOU_THRESH"])
print("DROPOUT_P:", FINAL_INFER_CONFIG["DROPOUT_P"])


In [ ]:
CONF_THRESH = FINAL_INFER_CONFIG["CONF_THRESH"]
NMS_IOU_THRESH = FINAL_INFER_CONFIG["NMS_IOU_THRESH"]

with torch.no_grad():
    print("Inference context ready.")


Experiment 10 Qualitative Error and Analysis

In [ ]:
print("CUDA available:", torch.cuda.is_available())
print("DEVICE:", DEVICE)
print("Model param device:", next(model.parameters()).device)

FINAL_CKPT = BEST_FINAL_CKPT if "BEST_FINAL_CKPT" in globals() and BEST_FINAL_CKPT else SELECTED_SWEEP_CKPT

if FINAL_CKPT is None or not os.path.exists(FINAL_CKPT):
    raise FileNotFoundError(f"Checkpoint not found: {FINAL_CKPT}")

print("Using checkpoint:", FINAL_CKPT)

_, val_loader, _ = get_dataloaders(16, S, B, C, augment=False)

DROPOUT_P = 0.3435466398976728
model, raw_model = build_sweep_model(DROPOUT_P)
raw_model.load_state_dict(torch.load(FINAL_CKPT, map_location=DEVICE))
model.eval()

CONF_THRESH_VALUES = [0.20, 0.25, 0.30, 0.35, 0.40, 0.50]
NMS_IOU_VALUES = [0.35, 0.40, 0.45, 0.50]

results = []

print("Sweeping inference thresholds on VALIDATION set")

for conf_thresh, nms_iou in itertools.product(CONF_THRESH_VALUES, NMS_IOU_VALUES):
    val_metrics, _ = evaluate_with_capture(
        model=model,
        loader=val_loader,
        conf_thresh=conf_thresh,
        iou_thresh=nms_iou
    )

    row = {
        "CONF_THRESH": conf_thresh,
        "NMS_IOU_THRESH": nms_iou,
        "val/mAP": val_metrics.get("val/mAP"),
        "val/mAP_50_95": val_metrics.get("val/mAP_50_95"),
    }
    results.append(row)
    print(row)

    if wandb.run is not None:
      wandb.log({
        "threshold_sweep/CONF_THRESH": conf_thresh,
        "threshold_sweep/NMS_IOU_THRESH": nms_iou,
        "threshold_sweep/val_mAP": val_metrics.get("val/mAP"),
        "threshold_sweep/val_mAP_50_95": val_metrics.get("val/mAP_50_95"),
    })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(["val/mAP", "val/mAP_50_95"], ascending=False).reset_index(drop=True)

display(results_df)

best_row = results_df.iloc[0].to_dict()

BEST_INFER_CONF_THRESH = float(best_row["CONF_THRESH"])
BEST_INFER_NMS_IOU_THRESH = float(best_row["NMS_IOU_THRESH"])

print("Best validation threshold pair:")
print(best_row)

THRESH_SWEEP_CSV = os.path.join(CKPT_DIR, "exp10_threshold_sweep_val.csv")
results_df.to_csv(THRESH_SWEEP_CSV, index=False)
print("Saved CSV ->", THRESH_SWEEP_CSV)


In [ ]:
# create data loaders
train_loader, val_loader, test_loader = get_dataloaders(16, S, B, C)

# get 5 actual image ids from the test subset
test_subset = test_loader.dataset
test_ids = [test_subset.dataset.img_ids[i] for i in test_subset.indices[:5]]

# use the tuned / balanced threshold
INFERENCE_CONF_THRESH = 0.5

NMS_IOU_THRESH = 0.2

# choose checkpoint:
# 1) best final retrained checkpoint if available
# 2) otherwise fallback to selected sweep checkpoint
FINAL_CKPT = BEST_FINAL_CKPT if "BEST_FINAL_CKPT" in globals() and BEST_FINAL_CKPT else SELECTED_SWEEP_CKPT

if not os.path.exists(FINAL_CKPT):
    raise FileNotFoundError(f"Checkpoint not found: {FINAL_CKPT}")

print("Using checkpoint:", FINAL_CKPT)

# build model with the same dropout used for the selected sweep / Exp 9
model, raw_model = build_sweep_model(0.3435466398976728)

raw_model.load_state_dict(torch.load(
    FINAL_CKPT,
    map_location=DEVICE
))
model.eval()

# run inference on 10 test images
for img_id in test_ids:
    img_path = f"{IMG_DIR}/{img_id}.jpg"
    inference(
        model=model,
        img_path=img_path,
        S=S, B=B, C=C,
        conf_thresh=INFERENCE_CONF_THRESH,
        iou_thresh=NMS_IOU_THRESH
    )